In [13]:
import json
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import time

from python_magnetrun.magnetdata import MagnetData
from python_magnetrun.MagnetRun import MagnetRun




DATA_DIR = Path("../Data for Stage")
PUPITRE_ROOT = Path("../pupitre_2023/srv-data-install")
PUPITRE_DIR = Path("../pupitre_2023/srv-data-install/M9")
PIGBROTHER = Path("../pigbrother_2025/M10_Overview_251201-0909.tdms")
DB = "magnetdb.duckdb"

FIELD_THRESHOLD = 0.1

In [2]:
# Sample the field at 20 points -> representation of the field profile
def compute_field_signature(field):

    values = field.to_numpy()

    if len(values) == 0:
        return ""
    
    idx = np.linspace(0, len(values) - 1, 20, dtype = int)
    signature = values[idx]

    return ",".join(f"{x:.2f}" for x in signature)

# HOUSING SUMMARY
### Load and merge the housing summary files


In [ ]:
rows = []
for file in sorted(DATA_DIR.glob("*_summary-*.json")):

    print(f"Loading {file.name}")

    site = file.stem.split("_")[0]
    year = int(file.stem[-4: ])

    with open(file, "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    df["site"] = site
    df["year"] = year

    rows.append(df)

summary_df = pd.concat(rows, ignore_index = True)

summary_df["experiment_id"]       = None
summary_df["field_max"]           = pd.Series(dtype = "float64")
summary_df["field_mean"]          = pd.Series(dtype = "float64")
summary_df["field_time_on"]       = pd.Series(dtype = "float64")
summary_df["mode"]                = ""
summary_df["field_signature"]     = ""
summary_df["reference_signature"] = ""

Loading M10_summary-2024.json
Loading M10_summary-2025.json
Loading M10_summary-2026.json
Loading M9_summary-2024.json
Loading M9_summary-2025.json
Loading M9_summary-2026.json


### Refresh the housing summary table and display basic stats

In [4]:
con = duckdb.connect(DB)
con.execute(
    """
        DROP TABLE IF EXISTS housing_summary
    """
)
con.register("summary_df", summary_df)
con.execute(
    """
        CREATE TABLE housing_summary AS
        SELECT *
        FROM summary_df
    """
)

print("\nRows:\n",
    con.execute(
        """
            SELECT site, year, COUNT(*) AS n
            FROM housing_summary
            GROUP BY site, year
            ORDER BY site, year
        """
    ).fetchdf()
)


Rows:
   site  year    n
0  M10  2024  146
1  M10  2025  233
2  M10  2026   66
3   M9  2024  162
4   M9  2025  191
5   M9  2026   57


### Data quality audit

In [5]:
## Check the date schema
print("\nTABLE SCHEMA: ",
    con.execute(
        """
            DESCRIBE housing_summary
        """
    ).fetchdf()
)
## Count imported records
print("\nNUMBER OF ROWS:", 
    con.execute(
        """
            SELECT COUNT(*) FROM housing_summary
        """
    ).fetchone()[0]
)
# Count number of records linked to experiments
print("NUMBER OF MATCHES:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.pupitre LIKE '%' || e.file
        """).fetchone()[0]
)


TABLE SCHEMA:              column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13                 site     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15      

In [6]:
# Check for missing files
print("\nMISSING FILES:\n",
    con.execute(
        """
            SELECT
                SUM(CASE WHEN overview = '' THEN 1 ELSE 0 END) AS overview,
                SUM(CASE WHEN archive  = '' THEN 1 ELSE 0 END) AS archive,
                SUM(CASE WHEN pupitre  = '' THEN 1 ELSE 0 END) AS pupiter,
                SUM(CASE WHEN trigger  = '' THEN 1 ELSE 0 END) AS trigger
            from housing_summary
        """
    ).fetchdf()
)
# Check for duplicate files
print("\nDUPLICATE FILENAMES:\n",
    con.execute(
        """
            SELECT filename, COUNT(*) AS n
            FROM housing_summary
            GROUP BY filename
            HAVING COUNT(*) > 1
            ORDER BY n DESC
        """
    ).fetchdf()
)


MISSING FILES:
    overview  archive  pupiter  trigger
0       0.0     10.0     85.0    767.0

DUPLICATE FILENAMES:
 Empty DataFrame
Columns: [filename, n]
Index: []


### Link with the user DB : Add foreign key column and populate

In [7]:
con.execute(
    """
        UPDATE housing_summary AS h
        SET experiment_id = e.id
        FROM experiments AS e
        WHERE h.pupitre LIKE '%' || e.file
    """
)

print("\nLINKED EXPERIMENTS:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
        """
    ).fetchone()[0]
)
print(
    con.execute(
        """
            SELECT experiment_id, filename, pupitre
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
            LIMIT 10
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT h.experiment_id, e.name, e.file, h.pupitre
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.experiment_id = e.id
            LIMIT 10
        """
    ).fetchdf()
)

rows = con.execute(
    """
        SELECT rowid, site, pupitre
        FROM housing_summary
        WHERE pupitre <> ''
    """
).fetchall()


LINKED EXPERIMENTS: 371
   experiment_id                  filename  \
0              1  M10_Overview_250313-1523   
1              2  M10_Overview_250313-1525   
2              4  M10_Overview_250313-1539   
3              5  M10_Overview_250315-1509   
4              9  M10_Overview_250315-1524   
5             10  M10_Overview_250316-1331   
6             11  M10_Overview_250316-2059   
7             12  M10_Overview_250317-0936   
8             13  M10_Overview_250408-1736   
9             15  M10_Overview_250410-1429   

                                             pupitre  
0  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
1  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
2  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
3  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
4  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
5  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
6  /mnt/LNCMIG-Data/records/srv-data-install/M10/...  
7  /mnt/LNCMIG-Data/records/

In [8]:
print(
    con.execute("""
        DESCRIBE housing_summary
    """).fetchdf()
)

            column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13                 site     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15        experiment_id 

In [9]:
start = time.perf_counter()
for i, (rowid, site, pupitre) in enumerate (rows, start = 1):

    filename = Path(pupitre).name
    filepath = PUPITRE_ROOT / site / filename

    if not filepath.exists():
        continue

    try: 

        md = MagnetData.fromtxt(str(filepath))
        df = md.getPandasData(None)

        field = df["Field"]
        field_signature = compute_field_signature(field)

        con.execute(
            """
                UPDATE housing_summary
                SET 
                    field_max = ?,
                    field_mean = ?,
                    field_time_on = ?,
                    field_signature = ?
                WHERE rowid = ?
            """, 
            (float(field.max()), float(field.mean()), int((field > FIELD_THRESHOLD).sum()), field_signature, int(rowid))
        )

        if i % 100 == 0:
            print(f"{i}/{len(rows)}")

    except Exception as e:
        
        print(filename, e)


end = time.perf_counter()
print(f"Dataframe updated in {int((end - start) // 60)} m {((end - start) % 60):.2f} s")

200/770
300/770
400/770
600/770
700/770
Dataframe updated in 2 m 43.74 s


In [10]:
# Validate update
print(
    con.execute(
        """
            SELECT COUNT(field_max) AS field_stats, COUNT(field_signature) AS signatures
            FROM housing_summary
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT experiment_id, field_max, field_signature
            FROM housing_summary
            WHERE field_signature <> ''
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            SELECT experiment_id, field_signature
            FROM housing_summary
            WHERE field_signature IS NOT NULL
            LIMIT 5
        """
    ).fetchdf()
)

   field_stats  signatures
0          446         855
   experiment_id  field_max                                    field_signature
0           <NA>    23.7965  0.00,0.00,0.44,2.97,5.62,8.12,10.65,13.30,15.8...
1           <NA>     0.0000  0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0...
2           <NA>     3.9981  0.00,4.00,1.00,1.00,1.60,1.60,1.60,2.50,2.50,2...
3           <NA>     0.9996  0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.37,0...
4           <NA>     8.7948  0.00,3.00,3.00,3.60,4.30,4.30,5.10,5.10,5.10,6...
5           <NA>     9.6640  0.00,0.00,2.23,9.59,9.59,9.59,9.59,9.59,9.59,9...
6           <NA>     4.4979  0.00,0.00,0.00,4.50,4.50,4.50,0.00,0.00,0.00,3...
7           <NA>     3.9981  0.00,0.00,4.00,4.00,4.00,4.00,4.00,4.00,4.00,4...
8           <NA>     6.5025  0.00,0.00,4.50,4.50,4.50,4.50,4.50,5.10,5.10,5...
9           <NA>    10.0039  0.00,7.30,7.30,7.30,7.30,7.94,8.20,8.20,8.20,8...
   experiment_id field_signature
0           <NA>                
1          

In [11]:
con.close()

files = sorted(PUPITRE_DIR.glob("*.txt"))

print(f"Found {len(files)} files")
file0 = files[0]
print(f"Loading: {file0.name}")

md = MagnetData.fromtxt(str(file0))
df = md.getPandasData(None)

print("\nAVAILABLE CHANNELS:", md.getKeys())
print("\nCOLUMNS:", df.columns.tolist())
print("\nFIRST ROWS:\n", df.head())

Found 821 files
Loading: 2023.01.30 - 16:02:56.txt

AVAILABLE CHANNELS: ['Date', 'Time', 'Field', 'Tin1', 'Tin2', 'Tout', 'TAlimout', 'HP1', 'HP2', 'BP', 'Flow1', 'Flow2', 'Rpm1', 'Rpm2', 'Idcct1', 'Idcct2', 'Idcct3', 'Idcct4', 'Icoil1', 'Ucoil1', 'DRcoil1', 'Tcal1', 'Icoil2', 'Ucoil2', 'DRcoil2', 'Tcal2', 'Icoil3', 'Ucoil3', 'DRcoil3', 'Tcal3', 'Icoil4', 'Ucoil4', 'DRcoil4', 'Tcal4', 'Icoil5', 'Ucoil5', 'DRcoil5', 'Tcal5', 'Icoil6', 'Ucoil6', 'DRcoil6', 'Tcal6', 'Icoil7', 'Ucoil7', 'DRcoil7', 'Tcal7', 'Icoil8', 'Ucoil8', 'DRcoil8', 'Tcal8', 'Icoil9', 'Ucoil9', 'DRcoil9', 'Tcal9', 'Icoil10', 'Ucoil10', 'DRcoil10', 'Tcal10', 'Icoil11', 'Ucoil11', 'DRcoil11', 'Tcal11', 'Icoil12', 'Ucoil12', 'DRcoil12', 'Tcal12', 'Icoil13', 'Ucoil13', 'DRcoil13', 'Tcal13', 'Icoil14', 'Ucoil14', 'DRcoil14', 'Tcal14', 'Icoil15', 'Ucoil15', 'DRcoil15', 'Tcal15', 'Icoil16', 'Ucoil16', 'DRcoil16', 'Tcal16', 'Pmagnet', 'Ptot', 'teb', 'tsb', 'debitbrut', 'Q']

COLUMNS: ['Date', 'Time', 'Field', 'Tin1', 'Tin2', '

# MODE INFERRING

In [26]:
mrun = MagnetRun.fromtdms(site = "M10", insert = "Overview", filename = str(PIGBROTHER))
mdata = mrun.getMData()
print(mdata)

print(mdata.Data["Courants_Alimentations"].columns)

magnetdata.fromtdms: ../pigbrother_2025/M10_Overview_251201-0909.tdms
magnetrun.fromtdms: start_time=2025-12-01 08:09:13.922483, type=<class 'datetime.datetime'>
MagnetData(Type=1, Groups={'Courants_Alimentations': {'Courant_A1': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A1', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A2': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A2', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A3': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A3', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A4': OrderedDict({'wf_start_time': np.datet

# PROPOSALS

In [52]:
# Load Proposals

proposals_df = pd.read_csv(DATA_DIR / "proposals.csv")
proposals_df["Debut"] = pd.to_datetime(proposals_df["Debut"], errors = "coerce")
proposals_df["Fin"]   = pd.to_datetime(proposals_df["Fin"], errors = "coerce")

print(proposals_df.head(), proposals_df.shape)

     Acronym  ProjectID ResearchArea  Facility ProposalType  accessMode  \
0  GMS06-217       3218           MS  Grenoble         EMFL         1.0   
1  GMS06-118       3494           MS  Grenoble         EMFL         1.0   
2  GMS04-118       3488           MS  Grenoble         EMFL         1.0   
3  GMA04-217       3322           MA  Grenoble         EMFL         1.0   
4  GSC01-118       3384           SC  Grenoble         EMFL         1.0   

   CallNumber  id ExperimentState  Site  ShotsHourDone  EnergyUsed      Debut  \
0         217  20            Done   M9i           60.0       531.0 2018-10-03   
1         118  21            Done   M9i           57.0       530.0 2018-11-30   
2         118  21            Done   M9i           44.0       159.0 2018-10-19   
3         217  20            Done   M9i           65.0       395.0 2018-10-23   
4         118  21            Done  M10e           55.0       200.0 2018-12-05   

         Fin  
0 2018-10-31  
1 2018-12-07  
2 2018-10-23  
3 

In [53]:
con = duckdb.connect(DB)

con.execute(
    """
        DROP TABLE IF EXISTS proposals
    """
)
con.register("proposals_df", proposals_df)
con.execute(
    """
        CREATE TABLE proposals AS
        SELECT *
        FROM proposals_df
    """
)

print(
    con.execute(
        """
            DESCRIBE proposals
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT * FROM proposals 
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            DESCRIBE experiments
        """
    )
)
print(
    con.execute(
        """
            SELECT * FROM experiments
            LIMIT 10
        """
    ).fetchdf()
)


        column_name   column_type null   key default extra
0           Acronym       VARCHAR  YES  None    None  None
1         ProjectID        BIGINT  YES  None    None  None
2      ResearchArea       VARCHAR  YES  None    None  None
3          Facility       VARCHAR  YES  None    None  None
4      ProposalType       VARCHAR  YES  None    None  None
5        accessMode        DOUBLE  YES  None    None  None
6        CallNumber        BIGINT  YES  None    None  None
7                id        BIGINT  YES  None    None  None
8   ExperimentState       VARCHAR  YES  None    None  None
9              Site       VARCHAR  YES  None    None  None
10    ShotsHourDone        DOUBLE  YES  None    None  None
11       EnergyUsed        DOUBLE  YES  None    None  None
12            Debut  TIMESTAMP_NS  YES  None    None  None
13              Fin  TIMESTAMP_NS  YES  None    None  None
       Acronym  ProjectID ResearchArea  Facility ProposalType  accessMode  \
0    GMS06-217       3218           MS

In [54]:
print(
    con.execute(
        """
            SELECT MIN(file), MAX(file), COUNT(*)
            FROM experiments
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT DISTINCT year
            FROM housing_summary
            ORDER BY year
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT MIN(Debut), MAX(Fin)
            FROM proposals
        """
    ).fetchdf()
)

                   min(file)                  max(file)  count_star()
0  2025.03.13 - 15:14:35.txt  2026.04.27 - 13:27:14.txt           755
   year
0  2024
1  2025
2  2026
  min(Debut)   max(Fin)
0 2009-01-19 2023-10-27


In [55]:
con.execute(
    """
        ALTER TABLE experiments
        ADD COLUMN IF NOT EXISTS proposal VARCHAR;
    """
)
con.execute(
    """
        UPDATE experiments e
        SET proposal = p.Proposal
        FROM proposals p
        WHERE e.start BETWEEN p.Debut AND p.Fin;
    """
)

BinderException: Binder Error: Table "e" does not have a column named "start"

Candidate bindings: : "status"

LINE 5:         WHERE e.start BETWEEN p.Debut AND p.Fin;
                      ^

In [ ]:
print(
    con.execute(
        """
            SELECT 
                COUNT(*) AS total_experiments,
                COUNT(proposal) AS linked,
                COUNT(*) - COUNT(proposal) AS missing
            FROM experiments
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT 
                start, end, file    
            FROM experiments
            WHERE proposal IS NULL
            ORDER BY start
            LIMIT 50
        """
    ).fetchdf()
)

ParserException: Parser Error: syntax error at or near "-"

LINE 3:                 COUNT(*) AS total-experiments,
                                         ^

In [ ]:
con.close()